In [ ]:
# Importing/installing necessary packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py # used for reading in galaxy image data
from PIL import Image
import glob
import os
import time

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, GridSearchCV
%matplotlib notebook

# Import the main ML package
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim

# Import the torchvision tools we'll be needing
import torchvision
from torchvision import transforms
from torchvision import datasets

# If you want to use built-in datasets instead of the one from Galaxy Zoo, uncomment the line below:
# from torchvision import datasets

# Image Classification with `PyTorch`

Especially useful for astronomy is the ability of AI to evaluate objects from images. This can be done using a Convolutional Neural Network (CNN) accessible within the `torchvision` module of `PyTorch`. In this example, we will practice building a simple image classification algorithm. The goal is to create a model that will accurately determine the classification of a given galaxy using computer vision.

The code for this exercise is adapted from [this tutorial](https://www.learnpytorch.io/03_pytorch_computer_vision/) by [Daniel Bourke](https://github.com/mrdbourke). For extra help, you may refer to the [PyTorch Classifier Tutorial](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html). The data are downloaded from the [Galaxy10 DECaLS dataset](https://astronn.readthedocs.io/en/latest/galaxy10.html), which was compiled from Galaxy Zoo. The labels are collated from volunteers (citizen scientists) tasked with classifying galaxies by-eye from randomly provided images from the SDSS DECaLS campaign. An example of sources under each label in the dataset is provided below:

![Example of galaxy classes from the DECals dataset.](https://astronn.readthedocs.io/en/latest/_images/galaxy10_example.png)

# Getting Started

Before creating the model, we first need to retrieve and clean up the data. You are encouraged to generate your own working method, but I will also provide some of mine for guidance.

The DECals dataset consists of an `.h5` file containing the galaxy images and labels corresponding to the galaxy types listed in the library below:

In [ ]:
# Run this for future use, or create your own

# Creating a dictionary to convert labels from integers
# to human-readable classifications from https://astronn.readthedocs.io/en/latest/galaxy10.html
galclass = {
    0: 'Disturbed',
    1: 'Merging',
    2: 'Round Smooth',
    3: 'In-between Round Smooth',
    4: 'Cigar Shaped Smooth',
    5: 'Barred Spiral',
    6: 'Unbarred Tight Spiral',
    7: 'Unbarred Loose Spiral',
    8: 'Edge-on without Bulge',
    9: 'Edge-on with Bulge'}

This dataset contains 17,736 galaxies, which takes a long time to read in and evaluate. There are several methods one may use to do this. For example, if you can preemptively split the images into classifications as you extract and save them, which will allow you to use the [ImageFolder()]() function `torchvision` to build the dataset you will train your model on. If you choose not to divide your images by classification, then you can define your own Dataset function to tell your model how to connect each image to a specific classification. The former method may be simpler, but for demonstrative purposes, we'll opt to define our own custom Dataset in the following exercises.

Below is an example of how to extract the galaxy images and organize them in a useful way. In the interest of time, I chose to compiled a list of 5000 randomly-drawn galaxies from the DECaLS galaxy file and saved them to a pandas DataFrame that tracks the the index of each galaxy, its label, and the corresponding classification. This DataFrame is saved as a `.csv` file in the main directory called `galaxy_classifications.csv`. I use this DataFrame to split the galaxies into a training set and a test set, each with their own directories and DataFrames, using `sklearn.model_selection.train_test_split()`. To keep my folders in order, I saved them in a directory called `galaxies`. All of these directories and subdirectories must be created either in your code or manually for the code below to run properly!

This is not the only (and possibly not the best) way to build your model! Feel free to create your own workflow as you work through this exercise!

## Objective:
*Extract the galaxy images from the Galaxy Zoo datafile and save them in a unique directory (you may choose to separate them by category). Keep track of the labels for each galaxy type.*

*The code below may be used as a starting point, but you may find it more instructive to come up with your own procedure.*

In [ ]:
### Example code I used to reduce the dataset and save images to a directory. It took ~1.5 minutes to run. ###
# To improve your model, try increasing the number of galaxies used (or use them all!)

# importing package for reading in the galaxy data file
import time

print("Starting data collection...")
starttime = time.perf_counter()

# Loading in the data from the file using h5py
# This may take a minute!
with h5py.File('Galaxy10_DECals.h5', 'r') as F:
    images = np.array(F['images']) # saves all images in file
    labels = np.array(F['ans'])    # saves all labels in file

### INPUT NEEDED ###
numgals =         # Sample size of galaxies

# Randomly sampling galaxies from the list and saving their images to a directory called 'galaxies'
randgalind = np.random.randint(0,len(labels), numgals).tolist()

### INPUT NEEDED ###
# Compiling and saving a DataFrame containing the index, label, and class of each galaxy selected above
randlabels =     # list of numeric labels of the selected galaxies
randclass =      # list of descriptive labels of the selected galaxies
rand_gals = pd.DataFrame(    # Input the galaxy indices, labels, and classifications
                        )
#rand_gals.sort_values(by="Index", inplace=True) # sort the DataFrame (optional)
rand_gals.to_csv(path_or_buf =       , # filename/path to save DataFrame to
                 index=False)

### INPUT NEEDED ###
# Splitting the sample into training and test samples
train_gals, test_gals = train_test_split(rand_gals,
                                         test_size=
                                        )
train_gals.to_csv(path_or_buf =       , # filename/path to save DataFrame to
                 index=False)
test_gals.to_csv(path_or_buf =       , # filename/path to save DataFrame to
                 index=False)

plt.ioff() # turning off the plotting mechanic for now

### INPUT NEEDED ###
# For each randomly-chosen index, save the training and test galaxy images in separate directories
for i in :  # pull image index from the train_gals DataFrame
    plt.figure()               # Hint: should you save them all with the same image dimensions?
    im = images[i].astype(int) # Pulls the correct galaxy image from the full catalog
    plt.imshow(im)             # Plots the image as a 3x3 figure
    plt.xticks([])
    plt.yticks([])
    # If you wanted to divide your galaxies into folders by classification,
    # this would be where you defined the folder names
    plt.savefig(fname=      , # what is the best way to organize your galaxy images?
                dpi=200,      # image resolution
                bbox_inches="tight", # cuts down on unnecessary whitespace
                pad_inches=-0.1)     # cuts down on unnecessary whitespace
    plt.close()

# Repeat for test galaxies
for i in :  # pull image index from the test_gals DataFrame
    plt.figure()               # Hint: should you save them all with the same image dimensions?
    im = images[i].astype(int) # Pulls the correct galaxy image from the full catalog
    plt.imshow(im)             # Plots the image as a 3x3 figure
    plt.xticks([])
    plt.yticks([])
    # If you wanted to divide your galaxies into folders by classification,
    # this would be where you defined the folder names
    plt.savefig(fname=      , # what is the best way to organize your galaxy images?
                dpi=200,      # image resolution
                bbox_inches="tight", # cuts down on unnecessary whitespace
                pad_inches=-0.1)     # cuts down on unnecessary whitespace
    plt.close()

endtime = time.perf_counter()
print(f"Total runtime for downloading {numgals} galaxies: {endtime - starttime} seconds")

# Reading in the Dataset
To build the training and test sets needed to create and evaluate the ML model, we must read in a dataset. If you divided your galaxies into folder by classification, you can do this by using [ImageFolder()](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html) from `torchvision`. Otherwise, you can define a custom dataset using [Dataset()](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Dataset).

Once the Dataset is defined, you can use [DataLoader()](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) from `PyTorch` to read it in. Then, we can use [random_split()](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.random_split) from `PyTorch` to split the data into a training and test set. There are different methods for doing this, so feel free to explore!

Since I chose not to divide the galaxies into folders by class, I define a Dataset below. Each file name in `galaxies` directory is in the form of `galaxies_{index}.jpg`, and their associated classifications are in the `galaxy_classification.csv` file under the "Label" column. The Dataset defined below pulls each image name and finds the appropriate label from the DataFrame. the `get_class_label` function should be editted in accordance with your unique workflow.

## Objective:
*Create a custom Dataset that works with your data organization. (You may use ImageFolder instead if you separated categories into directories). If you get stuck, check out [this tutorial on building custom datasets](https://www.learnpytorch.io/04_pytorch_custom_datasets/).*

In [ ]:
# Create a custom dataset using PyTorch's Dataset
# NOTE: The code below works with the way I organized my data.
# If you did it differently, you will need to edit accordingly

class GalSet(Dataset):
    def __init__(self, image_df, image_paths, transform=None):
        """
        Reads in the image file path and applies any input transformations.
        Upon initialization, defines the classification DataFrame, image_path,
        and any transformations.

        PARAMETERS
        -----------
        image_df (str)      : Name of the DataFrame containing the information about your galaxies in the dataset
        image_paths (str)   : Directory path pointing towards where the images are saved
        transform           : Object containing the transformations to apply to the data
                              (e.g. ToTensor(), Resize(), Normalize(), etc)

        RETURNS
        ----------
        Each index represents a single galaxy. If an index is called on the created object, the object
        will return the transformed image, numeric label, and label description of the corresponding galaxy.

        """

        self.image_labels = pd.read_csv(image_df)
        self.image_paths = image_paths
        self.transform = transform

    def __getitem__(self, index):
        """
        When a specific index of the Dataset object is used, return the associated
        image, label, and description of that galaxy.
        """
        ### INPUT NEEDED ###
        # Finding the image (this may look different for you if you changed your image naming convention)
        image_path = os.path.join(self.image_paths,
                                  # Galaxy file name, however you decided to name it
                                 )
        # Opening the image as an Image object
        image = Image.open(image_path).convert("RGB")
        # Pulling the associated classification label
        label = self.image_labels.iloc[index, 1]
        # Pulling the associated description label
        desc = self.image_labels.iloc[index, 2]
        # Apply any transformation defined
        if self.transform:
            image = self.transform(image)

        return image, label, desc

    def __len__(self):
        """Returns the length of the dataset. Should equal the number of images read in."""
        return len(self.image_labels)

## Objective:
*Read in the data using your custom Dataset. You may want to consider what kind of transformations besides `ToTensor()` may result in a good, flexible model.*

In [ ]:
### INPUT NEEDED ###
# Composing a transform to conver the image into a PyTorch tensor
# You may choose to use more than one transform; just add them to the list within Compose()!
tf = transforms.Compose([transforms.ToTensor(),
                        # Insert any other transformations you want to apply!
                        ])

### INPUT NEEDED ##
# Create the dataset using the custom Dataset we created. Alternatively, can use ImageFolder(), if applicable.
trainset = GalSet(image_df= ,
                  image_paths= ,
                  transform=tf)

testset = GalSet(image_df= ,
                 image_paths= ,
                 transform=tf)

Always visually inspect the data to ensure everything works as expected. Below is some code that will show an image when a dataset is read in.


## Objective:

*Write a code that will plot one galaxy of each category, similar to the image at the start of this Notebook. You can use the code below as a starting point, or start from scratch.*

*(If you get stuck, you may continue to training, as long as you are satisfied that you've labelled your galaxies correctly.)*

In [ ]:
### EXAMPLE CODE ###
#plt.ion() # turning the plotting mechanic back on

#def showimage(img):
#    """Helper function for plotting the transformed image as a color image."""
#    npimg = img.numpy()
#    plt.figure(figsize=(5,5))
#    plt.imshow(np.transpose(npimg, (1, 2, 0)))
#    plt.show()
#
## This code will plot a single galaxy/label pair. Can you get it to print an image for each of the 10 categories?
#dataiter = iter(dataset)
#images, labels, desc = next(dataiter)
#showimage(images)
#print(desc)

In [ ]:
# Your code here

# Creating the training and test set
Once the dataset is obtained and the read-in is confirmed to be working correctly, we can now load the data in with `DataLoader()` and split the data into a training and test set using `random_split()`.

## Objective:
*Create the training and test set. How many of each class exists within the training set? How do you expect this might bias your ML algorithm?*

In [ ]:
### Build a dataloader for your training set (not all parameters here may be necessary) ###
trainloader = DataLoader(dataset = , # dataset to load
                         batch_size = , # Number of galaxies in batch to train in chunks
                         shuffle = ,  # Boolean, reshuffle data after every round of training?
                         sampler = ,  # stragety for drawing samples from set
                         batch_sampler = , # returns batch of indices
                         num_workers = , # number of subprocesses to use for loading
                         collate_fn = , # merges a list of samples to form a mini-batch
                         drop_last = , # Boolean, drop last batch if incomplete?
                         timeout = , # timeout for collecting batches for workers
                         generator = , # RNG for generating indexes and generate base_seed for workers
                         persistent_workers= # Boolean, keep worker processes active after complete?
                        )

### Build a dataloader for your test set ###
testloader = DataLoader(dataset = , # dataset to load
                        batch_size = , # Number of galaxies in batch in chunks
                        shuffle = ,  # Boolean, reshuffle data?
                        sampler = ,  # stragety for drawing samples from set
                        batch_sampler = , # returns batch of indices
                        num_workers = , # number of subprocesses to use for loading
                        collate_fn = , # merges a list of samples to form a mini-batch
                        drop_last = , # Boolean, drop last batch if incomplete?
                        timeout = , # timeout for collecting batches for workers
                        generator = , # RNG for generating indexes and generate base_seed for workers
                        persistent_workers= # Boolean, keep worker processes active after complete?
                        )

### Check how many of each type of galaxy exist within the training/test sets ###

# Building a Convolutional Neural Network

Now we're ready to create the model -- specifically, a convolutional neural network (CNN). CNNs basically run images through several different layers that alter the data with the intent of learning something about the data's features, represented below:  

<img src="https://www.mathworks.com/discovery/convolutional-neural-network/_jcr_content/mainParsys/band_copy_copy/mainParsys/lockedsubnav/mainParsys/columns/a32c7d5d-8012-4de1-bc76-8bd092f97db8/image_792810770_copy.adapt.full.medium.jpg/1745491656387.jpg" width="600"/>

Image credit: [MathWorks](https://www.mathworks.com/discovery/convolutional-neural-network.html#)



It starts by feeding in the input layer (AKA, the images you want to train the model on). In our case, we're using full-color RGB images, so the input layer will have 3 channels for the red, green, and blue filter. Then, it runs the input layer(s) through a series of hidden layers, including the convolutional layer, activation layer, and pooling layer. Simplistically, these layers contain learned weights (AKA kernels) that are used to determine what features within each image most heavily determines their classification. The images are iterated through these layers until some pre-determined criterion is met. The data are then released as outputs. For a more in-depth explanation of CNNs, [check out this GitHub webpage](https://poloclub.github.io/cnn-explainer/).

The example CNN below is adapted from [the PyTorch Classifier tutorial](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) and the [PyTorch Computer Vision](https://www.learnpytorch.io/03_pytorch_computer_vision/#7-model-2-building-a-convolutional-neural-network-cnn) tutorial. This will likely result in a sub-optimal model, but may be a good place to start if you don't know what to try first! Explore how adding different steps and changing their order affects your model output.


In [ ]:
# import torch.nn as nn
# import torch.nn.functional as F
#
# # Defining an ultra-simple neural network that takes 3-channel (RGB) images
# # How would you improve this?
#
# class myCNN(nn.Module):
#     """
#     hidden_units:   number of neurons in the hidden layer
#     output_shape:   one output neuron per class in the dataset
#     """
#     def __init__(self, hidden_units: int, output_shape: int):
#         super().__init__()
#         # Runs data through a convolutional layer for a 3-channel, 2D image, a ReLU activation layer, and a max pooling layer
#         self.block_1 = nn.Sequential(nn.Conv2d(in_channels=3, # one channel per color
#                                                out_channels=hidden_units, # channels produced by convolution
#                                                kernel_size=3, # convolving kernel size
#                                                stride=1, # convolving kernel step size (default = 1)
#                                                padding=0), # margins added to image)
#                                      nn.ReLU(),
#                                      nn.MaxPool2d(kernel_size=2,
#                                                   stride=2)
#                                     )
#         self.classifier = nn.Sequential(nn.Flatten(), # flattens inputs into single vector
#                                         nn.LazyLinear(out_features=output_shape) # Can use nn.Linear, but will need to know the input shape.
#                                        )
#
#     def forward(self, x):
#         x = self.block_1(x)
#         x = self.classifier(x)
#         return x

## Objective:
*Build your own CNN. You may use the example above as a starting place, but it is a poor model! Consider what changes you could make to improve it*

# Define a Loss Function and Optimizer


Once the model is created, it's a good idea to try to optimize the learning rate, the step size used by the neural network as it learns from the training data. If the learning rate is set too low, then the CNN will progress very slowly, but set it too high and it will overshoot and result in poorly fit models.

<img src="https://miro.medium.com/v2/resize:fit:720/format:webp/1*wfx8jLOyAmtsJpti1q_JMw.png"/>

Image credit: [Jeremy Jordan](https://www.jeremyjordan.me/nn-learning-rate/)

An optimal learning rate can be estimated by plotting the loss function -- representing how often the model misidentifies target values -- as a function of the learning rate. Generally, at some point the loss function sharply declines with increasing learning rate. The ideal learning rate is somewhere in this zone. You can get practice manually finding the optimal learning rate if you follow the `fastai` tutorial at the end of the notebook.

For now, we can define the optimizer and loss function we'll use for our model. For the loss function, `nn.CrossEntropyLoss()` is one option for measuring how far the prediction from your CNN is from the actual label. `optim.SGD` (stochastic gradient descent) is used to optimize the model by updating the parameters in such a way that (hopefully) minimize the loss function. You may start by defining some learning rate (`lr`), but you may choose to change this if you feel the algorithm is still poorly optimized.

## Objective:
*Initiate your model, a loss function, and an optimizer based on your model.*

In [ ]:
### Initiate the model to train ##
cnn = <your_CNN_name>(hidden_units= , # number of neurons in the hidden layer
                      output_shape= ) # one output neuron per class in the dataset

### Set up the loss function and optimizer ###
loss_fn = nn.CrossEntropyLoss() # measures how far from the correct answer the prediction is
optimizer = optim.SGD(params = cnn.parameters(),  # implements stochastic gradient descent
                      lr= , # learning rate
                      momentum= # "momentum", helps model get out of local minima
                     )

# Is there a better loss function or optimizer to use instead of the examples above?

# Training the Model

To train the model, we will want to read every image (or batch of images) from the training set (trainloader) into the CNN and run the set through at least a couple of epochs. Here, an epoch refers to the number of times the dataset is looped through the CNN. In theory, more epochs should lead to a stronger model, but there is a limit to how beneficial additional epochs are for the model. With a limited dataset, you don't want to overfit the model!

The general steps for training the model include:
* Activate training mode with `<model>.train()`
* For each epoch, zero out your optimizer gradient with `<optimizer>.zero_grad()`
* Run each image or batch of images through the model
* Calculate a loss (`<loss>`) by running your predicted and true labels through your loss function
* Update the weights in your model via backpropagation with `<loss>.backward()`
* Update your model parameters with `<optimizer>.step()`
* Repeat for the next image or batch of images
* Once the model has been trained on all images or batches, repeat for each additional epoch.

If your model is training correctly, you should notice that each epoch takes some time to complete (possibly several minutes). You should print your losses at each step to evaluate in real-time whether or not your model is improving. If you notice your losses aren't improving (decreasing), then you may need to change the learning rate or momentum in your optimization function.

## Objective:
*Train your model on the data. Print the results from the loss function to evaluate the effectiveness of the model as it being trained. You may wish to try different learning rates and momentum values in your optimizer, or increase/decrease the number of epochs over which the model is trained.*

In [ ]:
cnn.train() # place model in training mode

print("Training model...")

### INPUT NEEDED ###
for epoch in range( # decide how many epochs you want to run the data through

    # Keeping count of the Classification Cross-Entropy loss
    total_loss = 0
    batch_loss = 0

    starttime = time.perf_counter() # for timing the processes

    # for each galaxy in the training set...
    for i, gal in enumerate(trainloader):

        image, label, desc = gal # pull data for galaxy batch

        ##################################
        ### INSERT TRAINING STEPS HERE ###
        ##################################

        ### Zero out the optimizer ###
        #
        ### Run the image through your CNN ###
        #
        ### Calculate the loss ###
        #
        ### Backpropagate the loss to update CNN weights ###
        #
        ### Update the optimizer ###
        #

        # Keeping track of losses, for real-time evaluation
        total_loss += loss.item() # update the loss counter
        batch_loss += loss.item()

        # Printing off losses after every batch, for real-time evaluation
        if i > 0 and i % batchsize == 0: # print stats after every so many batches
            print(f"Processed {i+1}/{len(trainloader)} batches; loss: {round(batch_loss,3)}")
            batch_loss = 0

    # End training epoch and print off stats
    endtime = time.perf_counter()
    print(f"Epoch {epoch + 1} completion time: {round((endtime - starttime)/60.,3)} minutes; total loss: {round(total_loss,3)}\n")


print("Training complete!")

Are your total losses at the end of training lower than at the beginning? If so, your model is on the right track!

# Evaluating the Model

We can now test the model by running the test set through the CNN. The predictions from the CNN will save as a list of probabilities telling you how likely it is that the image has a given label (0 to 9). The index with the highest probability is the label that the model assigns to the image. This should be compared to the actual labels to determine how accurate your model is.

In [ ]:
# Loading up the first batch only
# Alternatively, you can use the full testset by setting the batch_size to the length of the test set when calling DataLoader
dataiter = iter(testloader)
test_imgs, test_labels, test_descs = next(dataiter)

cnn.eval() # puts model in evaluation (test) mode

predicts = cnn(test_imgs) # probablities per object
_, predicts_index = torch.max(predicts, 1) # converting probabilities to indices
predict_labels = [i.item() for i in predicts_index] # converting indices to simple list of labels
predict_descs = [galclass[i.item()] for i in predicts_index] # converting indices/labels to descriptions

It is useful to visually inspect the images that the model was given.

## Objective:
*Build a function for plotting the images and the labels assigned. Compare the predicted labels to the actual labels. Did your model do a good job?*

In [ ]:
### Build your function here ###
def plotGals( # function parameters
    """
    Plots the galaxies and labels or descriptions fed into the model and compares it
    to the predicted labels.

    PARAMETERS
    ----------

    """

In [ ]:
# Test your function on test_imgs, test_labels, and/or test_descs
plotGals(<insert appropriate parameters>)

If you used a model similar to my ultra-simplistic CNN, then it probably looks like your model did a terrible job. Look back at the histogram you made of how many galaxies of each classification appear in your training set. Did that seem to affect your results here?

**How else can we evaluate the model to determine its shortcomings?**|

A very useful tool for model analysis is a confusion matrix. A confusion matrix plots the number of images with each classification against the predicted classification assigned by the model:

![Example of a confusion matrix.](https://scikit-learn.org/stable/_images/sphx_glr_plot_confusion_matrix_001.png)

A model that perfectly predicts the actual classification of each object in the data set will result in a confusion matrix with zeros everywhere except along the top-left-to-bottom-right diagonal. In reality, all models will have some degree of error, but a good model will have the highest values where the actual classification meets the predicted classification.

## Objective:
*Create a confusion matrix using tools from `sklearn.metrics` or other similar packages.*

In [ ]:
### INPUT NEEDED ###
cmatrix = confusion_matrix(y_true = , # list of actual numerical labels
                           y_pred = , # predicted numerical labels
                           labels= # list of possible numerical labels
                          )
CMDisplay = ConfusionMatrixDisplay(confusion_matrix = cmatrix,
                                   display_labels = # list of label descriptions associated with the numeric labels
                                  )
CMDisplay.plot(cmap=plt.cm.Reds)
plt.xticks(rotation=60, ha='right')
plt.tight_layout(pad=1)
plt.show()

How else can we visualize what's going on with the model? One way may be to plot the weights being used by the model to determine which features within the image leads to one classification scheme over another. This process is called **feature extraction**. In the code below, I pull out each convolutional layer from my CNN and examine what features they see in the input image.

The code below is adapted from [this video tutorial](https://www.youtube.com/watch?v=Q6YHaadg3MI).

In [ ]:
model_children = list(cnn.children()) # extracts information about each step of the model
print(model_children) # You should see a list containing all of the layers you put into your model

conv_layers = [] # keeps track of our convolutional layers

for child in model_children:
     # Since I embedded my layers inside a Sequential method, each layer in each child will have its own children!
    if type(child) == nn.Sequential:
        for layer in child.children():
            if type(layer) == nn.Conv2d:
                conv_layers.append(layer)

print(conv_layers)

# Select an image from the test set and print it
img_ind = random.randint(0,len(test_imgs)+1) # selecting random galaxy
img = test_imgs[img_ind]
npimg = img.numpy()

plt.ioff()
plt.figure()
plt.imshow(np.transpose(npimg, (1, 2, 0)))
plt.title(test_descs[img_ind])
plt.axis('off')
plt.show()

img_unsq = img.unsqueeze(0)
results = [conv_layers[0](img_unsq)] # creates a feature map from the image and first convolutional layer

# If there are additional convolutional layers, then you will want to apply each layer to the results from the
# previous layer
if len(conv_layers) > 1:
    for i in range(1,len(conv_layers)):
        results.append(conv_layers[i](results[-1]))

outputs = results

plt.ioff()
for i, layer in enumerate(outputs):
    plt.figure()
    layer_vis = layer.squeeze()
    print(f"Layer {i}")
    for j, f in enumerate(layer_vis):
        plt.subplot(2,5,j+1)
        plt.imshow(f.detach().cpu().numpy())
        plt.axis("off")
    plt.show()

print(test_descs[img_ind])

## Objective:
*Create a feature map for each of the 10 galaxy classifications in your dataset. Can you tell what features are being highlighted as the most important for each classification type?*

In [ ]:
### Your code here ###

# Saving and Loading Your Model

If you're happy with your model or want to save it to compare to other models later, you can save/load it using the code below.

In [ ]:
### OPTIONAL ###

# Saving the model
torch.save(obj=cnn.state_dict(), # only saving the state_dict() only saves the learned parameters
           f= # path and name to save file
          )

# To load the model, set up the CNN and load in the state dictionary saved above
model = <your_CNN_name>(hidden_units= ,
                        output_shape=
                       )

model.load_state_dict(torch.load(f= ,# path to the saved model state dictionary
                                 weights_only=True)
                     )

# Model Improvements
Did your model perform poorly? If so, you may make changes to the CNN, optimizer, training, or even the image transformations you begin with to attempt to improve it. What combination of strategies results in the best model?

## Objective:
*Build a better model and compare the results.*

In [ ]:
### Your code here ###
# Test it out as you seem fit. Be sure to do a thorough evaluation!

# (OPTIONAL) Test your model on more galaxy images
Saved in the `test_galaxies` directory are three galaxy images. Can your model tell what kind of galaxies these are? You can also find images of different galaxies on the Internet and test your model further. (You may need to resize these images to make them work in your model, depending on how the model is built).

In [ ]:
### Your code here ###

# (OPTIONAL) Other resources and exercises

There is a Python package called `fastai` that is built upon `PyTorch` to do much of the same things we do in this workshop. If you're interested in learning more, check out and follow the steps in [this tutorial](https://colab.research.google.com/github/jwuphysics/AstroHackWeek2022/blob/main/day2_ml_tutorial/02-deep-learning.ipynb#scrollTo=gzG6d5DhMEy5) by [John F. Wu](https://jwuphysics.github.io/resources/). Can you replicate your results using `fastai`? What benefits (or weaknesses) do you notice?

In [ ]:
# fastai is a high-level deep learning library built on Pytorch.
# If not already installed, install with the code below.
# Otherwise, comment out
# !pip install -q fastai --upgrade

import fastai
from fastai.vision.all import *
from fastai.vision.data import *

The steps below are just to demonstrate a few basic `fastai` utilities.

To build the model, we must create a DataLoader object, which we will do using `ImageDataLoaders`. In theory, there are several ways to do this: we can build the DataLoaders from a DataBlock, a DataFrame, or a folder. However, I tested several methods and, despite my best efforts, I kept running into a mysterious indexing error for all but the DataBlock method. Below, I'll include my broken code in case you want to take a crack at it, but for the remainder of this demonstration, I'll use the working DataBlock method.


In [ ]:
# gals = pd.read_csv("galaxy_classifications.csv") # DataFrame containing ALL galaxies
#
# #### Attempting to create a DataLoader using .from_folder to split the training/test sets ####
# ImageDataLoaders.from_folder(path=f"{os.getcwd()}/galaxies",
#                             train = "train",
#                             valid = "test")
#
# #### Attempting to create DataLoader from a DataFrame containing all image paths and validation boolean ####
# # Building a DataFrame containing training and test set image paths,
# # labels, and whether or not it is in the validation set.
# gal_paths = [f"{os.getcwd()}/galaxies/train/galaxy_{g}.jpg" for g in train_gals["Index"].values.tolist()] + [f"{os.getcwd()}/galaxies/test/galaxy_{g}.jpg" for g in test_gals["Index"].values.tolist()]
# gal_labels = [c for c in train_gals["Class"].values.tolist()] + [c for c in test_gals["Class"].values.tolist()]
# gal_valid = ["False" for c in train_gals["Class"].values.tolist()] + ["True" for c in test_gals["Class"].values.tolist()]
# gal_data = {"image_path": gal_paths, "label": gal_labels, "is_valid": gal_valid}
# # Building DataFrame
# gal_dataset = pd.DataFrame(data = gal_data)
#
# dataloaders = ImageDataLoaders.from_df(gal_dataset,
#                                        fn_col = "image_path", # column in DataFrame pointing to the image filename
#                                        label_col="label",   # column in DataFrame pointing to labels
#                                        valid_col="is_valid", # column in DataFrame pointing to whether train/test
#                                       splitter=ColSplitter("is_valid"))
#
# #### Attempting to split the DataBlock into training and test sets by our pre-determined directories ####
# dblock = DataBlock(blocks=(ImageBlock, CategoryBlock),
#                    get_x = ColReader(["Index"],      # Header in classification DataFrame containing index differentiating between galaxy image names
#                                      pref="galaxy_", # filename prefix before the galaxy index
#                                      suff=".jpg"),    # filename suffix after the galaxy index
#                    get_y = ColReader(["Class"]),     # Header in classification DataFrame containing the label
#                    splitter=GrandparentSplitter(train_name="galaxies/train", # Function for determining the training and test sets
#                                                 valid_name="galaxies/test")
#                   )
#
# # Loading the data with data loaders
# # ImageDataLoaders is a function from fastai.vision.data
# dataloaders = ImageDataLoaders.from_dblock(dblock, gals)

First, we need to set up a DataBlock, which will contain all of the information we will use to create our ML classification model. In the cell below, the defined DataBlock will pull information from a pandas DataFrame containing all galaxies in our dataset, which we will call `gals`.

The `blocks` parameter lets the algorithm know what kind of data it should expect for the features and labels. In this case, out input features are images (`ImageBlock`), and our desired outputs are classification labels (`CategoryBlock`). If we wanted to use this model to predict a feature value, we would use `RegressionBlock` for the output/labels parameter instead.

We must also define where the code should get the X and y values (e.g. the features and labels), which is done through `get_x` and `get_y`.

`get_x` pulls the name of the images of each galaxy, which are being used in place of the features. Here, I compile the filenames from the `gals`, where each file name is in the form of `galaxies_{index}.jpg`.

For `get_y`, the labels are pulled from the `Class` header in the `gals` DataFrame. We could also pull from the `Label` column to use the numeric labels instead; the results of the model fitting would not change.

`DataBlock` can also be set to split the data into training and test sets using the `splitter` parameter. If all of your galaxies are in a single folder, you can use `splitter=RandomSplitter()` to split the sets up randomly, where the value in `RandomSplitter` is the fractional size of the test/validation set. In theory, you should be able to tell `DataBlock` that your data is already split into subdirectories by using the `GrandparentSplitter()` function (see the example code above), but since that's not working, I opted instead to save all of my galaxies into a new directory called `all`.

In [ ]:
# ImageBlock and CategoryBlock are functions from fastai.vision.data
# ColReader and RandomSplitter are from fastai.data.transforms

### INPUT NEEDED ###
gals = pd.read_csv(   ) # DataFrame containing ALL galaxies

### INPUT NEEDED ###
dblock = DataBlock(blocks=(ImageBlock, CategoryBlock),
                   get_x = ColReader(cols= ,   # Column in classification DataFrame containing galaxy index (given as a list)
                                     pref= ,   # filename prefix before the galaxy index
                                     suff= ),  # filename suffix after the galaxy index (e.g. .jpg)
                   get_y = ColReader(cols= ),  # Header in classification DataFrame containing the label
                   splitter=RandomSplitter( )  # Set train/test split
                  )
### NOTE FOR THE CODE ABOVE ###
# get_x will read as the file path and name of the galaxy image
# in the format {pref}{index from cols}{suff}. For example, if {pre} = "galaxies/train/galaxy_",
# {cols} = ["Index"], {suff} = ".jpg", DataBlock will look for files with the name format:
# "galaxies/train/galaxy_{index}.jpg"

# Loading the data with data loaders
# ImageDataLoaders is a function from fastai.vision.data
dataloaders = ImageDataLoaders.from_dblock(dblock, gals)

The data is loaded in using `ImageDataLoaders`, which takes the DataBlock and the DataFrame it was built with to create an object containing the loaded images/data.

As always, it's important to inspect the data and get familiar with it. Notice how easy this is to plot images with `fastai`, compared to having to write your own function to do the same.

In [ ]:
### INPUT NEEDED ###
# Showing an example of the galaxies loaded
dataloaders.show_batch(nrows= , # number of rows in plotted figures
                       ncols= ) # number of columns

Now we're ready to create the model. The CNN model we're using is called `vision_learner` (formerly known as `cnn_learner`)

Once the model is created, we can start optimizing the learning rate, as discussed previously. Remember: if the learning rate is set too low, then the CNN will progress very slowly, but set it too high and it will overshoot and result in poorly fit models.

<img src="https://miro.medium.com/v2/resize:fit:720/format:webp/1*wfx8jLOyAmtsJpti1q_JMw.png"/>

Image credit: [Jeremy Jordan](https://www.jeremyjordan.me/nn-learning-rate/)

An optimal learning rate can be estimated with the `lr_find` function. This function plots the loss function -- representing how often the model misidentifies target values -- as a function of the learning rate. Generally, at some point the loss function sharply declines with increasing learning rate. The ideal learning rate is somewhere in this zone.

Run the code below and interpret the ideal learning rate. How does this compare to the learning rate you selected for your model(s) before?

In [ ]:
# Creating the learner/model

# a "learner" evaluates and updates the model
# resnet18 and error_rate comes from fastai, as do the metrics below
vis_model = vision_learner(dataloaders, resnet18, metrics=[error_rate, accuracy])

# find an optimal learning rate. Are there other functions that would be good to see?
lrs = vis_model.lr_find(suggest_funcs=(minimum, steep, valley, slide))
print(lrs)

`vis_model` is our model, which we set up as a [Resnet18 model](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html). As of right now, we haven't done any actual training. We can see how well an untrained model does by plotting a confusion matrix for the training data. `fastai` contains a utility called `ClassificationInterpretation`, within which is the `plot_confusion_matrix()` tool.

In [ ]:
# Plotting the initial confusion matrix
interp = ClassificationInterpretation.from_learner(vis_model)
interp.plot_confusion_matrix()

In addition to the confusion matrix, `fastai` can also show you which classifications the model is struggling with most using the `plot_top_losses` function. This is just an additional visualization tool that can help you understand what the model may be missing. This tool posts the predicted and actual labels it is most often mischaracterizing, the loss, and the probability (how confidently wrong the model is). The top losses should reflect what the confusion matrix is showing.

In [ ]:
### INPUT NEEDED ###
# Plotting the top losses (misclassifications)
interp.plot_top_losses(k= , # number of losses to plot
                       nrows= ) # formatting the plot

We can begin our initial round of training using the `fit_one_cycle`command and inputting the number of epochs over which to train the model, and the maximum learning rate. For the learning rate, you can either input a value manually, or select one of the points from the learning rates obtained with `lr_find` (e.g. `lrs.steep`, `lrs.slide`, `lrs.valley`, `lrs.minimum`, or any other you can obtain from the `lr_find` function).

After training the model, create another confusion matrix to determine whether your model has improved, and/or whether it requires additional training.

In [ ]:
### INPUT NEEDED ###
# Fitting and updating the model over a cycle of 3 epochs
vis_model.fit_one_cycle(n_epoch= , # number of training epochs
                        lr_max =  # maximum learning rate (optimal rate should not exceed this value)
                       )

In [ ]:
# Plotting the initial confusion matrix
interp = ClassificationInterpretation.from_learner(vis_model)
interp.plot_confusion_matrix()

You should notice an improvement, but more likely than not, your model is not yet to the point that you want it to be. You can choose to run additional training rounds on the model using `fit_one_cycle` or `fit`, or you can attempt to fine-tune the model parameters with `fine_tune`, as demonstrated below.

In [ ]:
### INPUT NEEDED ###
# Fine-tuning the model through retraining over multiple epochs
vis_model.fine_tune(epochs=  , # number of epochs. Try a higher number than before
                    base_lr= ) # choose a learning rate

Evaluate with a confusion matrix and/or `plot_top_losses`.

In [ ]:
# New confusion matrix
interp = ClassificationInterpretation.from_learner(vis_model)
interp.plot_confusion_matrix()

In [ ]:
# Plotting the top losses (misclassifications)
interp.plot_top_losses(k = , # number of galaxies to plot
                       nrows = # number of rows to plot
                      )

Once the model is in good condition, one can then start using it to classified unknown images:
```vis_model.predict({image name})```

Try it on other galaxies in the `.h5` data file or on galaxy images you find elsewhere! A couple of test images are provided in the `test_galaxies` directory.

In [ ]:
### Your code here ###